In [1]:
import sys
from tqdm import tqdm
import numpy as np
import os
import pandas as pd
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score
from torch.utils.data import Dataset
import time
from torch.utils.data import DataLoader
import json
from sklearn.model_selection import train_test_split
import argparse
from torch import optim
import pandas as pd
import numpy as np
import os
# import seaborn as sns
from tqdm.auto import tqdm
import warnings
from sklearn.preprocessing import OneHotEncoder
import gc
import pickle
from sklearn.decomposition import TruncatedSVD
import glob
from torch.nn.utils.rnn import pad_sequence
import math
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv
import torch_geometric
from sklearn.calibration import LabelEncoder
import ast

DATA_PATH = r"C:\Coding\mabe\data"
CACHE_PATH = r"C:\Coding\mabe\MABe_2022_TVAE\cache"
SPECIAL_VALUE = -99999.0
CHUNK_SIZE = 1000
SUB_SEQ_LENGTH = 21
SLIDING_WINDOW = 1
ALPHA = 10

True
NVIDIA GeForce RTX 4070 SUPER


In [2]:
import pandas as pd

# Your DataFrame (assume it's called df)

# Step 1: Define the priority mapping
priority_map = {
    'top_left': [
        ('ear_left', 1),
        ('forepaw_left', 2),
        ('headpiece_bottombackleft', 3),
        ('headpiece_bottomfrontleft', 4),
        ('headpiece_topbackleft', 5),
        ('headpiece_topfrontleft', 6),
        ('lateral_left', 7),
    ],
    'top_right': [
        ('ear_right', 1),
        ('forepaw_right', 2),
        ('headpiece_bottombackright', 3),
        ('headpiece_bottomfrontright', 4),
        ('headpiece_topbackright', 5),
        ('headpiece_topfrontright', 6),
        ('lateral_right', 7),
    ],
    'bottom': [
        ('tail_base', 1),
        ('tail_midpoint', 2),
        ('tail_middle_1', 3),
        ('tail_middle_2', 4),
        ('hindpaw_left', 5),
        ('hindpaw_right', 6),
        ('hip_left', 7),
        ('hip_right', 8),
        ('tail_tip', 9),
    ],
    'top_center': [
        ('nose', 1),
        ('head', 2),
        ('neck', 3),
        ('body_center', 4),
    ],
}

# Flatten for quick lookup
bodypart_to_target = {}
for target, parts in priority_map.items():
    for part, priority in parts:
        bodypart_to_target[part] = (target, priority)

# Step 2: Filter only relevant bodyparts (ignore 'spine_1', 'spine_2', etc.)
def reassign_points(path, start, stop):
    df = pd.read_parquet(path)
    # df['video_frame'] = df['video_frame'] - df['video_frame'].iloc[0]
    # print(f"shape before: {df.shape}")
    # Filter early: only keep frames within the desired range
    df = df[(df['video_frame'] >= start) & (df['video_frame'] < stop)]
    # print(f"shape after: {df.shape}")

    # df['assignment'] = df['bodypart'].map(bodypart_to_target)

    # # Drop rows with no assignment
    # df = df.dropna(subset=['assignment'])

    # # Add target and priority columns
    # df[['target', 'priority']] = pd.DataFrame(df['assignment'].tolist(), index=df.index)

    # # Keep only the highest priority (lowest number) per group, per target
    # df_sorted = df.sort_values(by=['video_frame', 'mouse_id', 'target', 'priority'])
    # df_deduped = df_sorted.drop_duplicates(subset=['video_frame', 'mouse_id', 'target'], keep='first')

    # Pivot to get final structure
    pivot_x = df.pivot(index='video_frame', columns=['mouse_id', 'bodypart'], values='x')
    pivot_y = df.pivot(index='video_frame', columns=['mouse_id', 'bodypart'], values='y')
    pivot_x.columns = [f"mouse_{m}_{bp}_x" for m, bp in pivot_x.columns]
    pivot_y.columns = [f"mouse_{m}_{bp}_y" for m, bp in pivot_y.columns]
    df_wide = pd.concat([pivot_x, pivot_y], axis=1).sort_index(axis=1)
    # print(df_final.shape)

    return df_wide

In [3]:
body_parts = [
    "ear_left",
    "ear_right",
    "tail_base",
    "nose",
    "neck",
    # "body_center",
    # "tail_tip",
    # "tail_midpoint",
    # "forepaw_left",
    # "forepaw_right",
    # "hindpaw_left",
    # "hindpaw_right",
    "hip_left",
    "hip_right",
    # "lateral_left",
    # "lateral_right",
    # "spine_1",
    # "spine_2",
    # "tail_middle_1",
    # "tail_middle_2",
    # "head",
    # "headpiece_bottombackleft",
    # "headpiece_bottombackright",
    # "headpiece_bottomfrontleft",
    # "headpiece_bottomfrontright",
    # "headpiece_topbackleft",
    # "headpiece_topbackright",
    # "headpiece_topfrontleft",
    # "headpiece_topfrontright"
]
# body_parts = ['top_center', 'top_left', 'top_right', 'bottom']
len(body_parts)

all_cols = []
for i in range(1, 5):
    for part in body_parts:
        for j in ['x', 'y']:
            all_cols.append(f"mouse_{i}_{part}_{j}")
print(all_cols)
final_cols = ['mouse_1_ear_left_x', 'mouse_1_ear_left_y', 'mouse_1_ear_right_x', 'mouse_1_ear_right_y', 'mouse_1_tail_base_x', 'mouse_1_tail_base_y', 'mouse_1_nose_x', 'mouse_1_nose_y', 'mouse_1_neck_x', 'mouse_1_neck_y', 'mouse_1_hip_left_x', 'mouse_1_hip_left_y', 'mouse_1_hip_right_x', 'mouse_1_hip_right_y', 'mouse_2_ear_left_x', 'mouse_2_ear_left_y', 'mouse_2_ear_right_x', 'mouse_2_ear_right_y', 'mouse_2_tail_base_x', 'mouse_2_tail_base_y', 'mouse_2_nose_x', 'mouse_2_nose_y', 'mouse_2_neck_x', 'mouse_2_neck_y', 'mouse_2_hip_left_x', 'mouse_2_hip_left_y', 'mouse_2_hip_right_x', 'mouse_2_hip_right_y', 'mouse_3_ear_left_x', 'mouse_3_ear_left_y', 'mouse_3_ear_right_x', 'mouse_3_ear_right_y', 'mouse_3_tail_base_x', 'mouse_3_tail_base_y', 'mouse_3_nose_x', 'mouse_3_nose_y', 'mouse_3_neck_x', 'mouse_3_neck_y', 'mouse_3_hip_left_x', 'mouse_3_hip_left_y', 'mouse_3_hip_right_x', 'mouse_3_hip_right_y', 'mouse_4_ear_left_x', 'mouse_4_ear_left_y', 'mouse_4_ear_right_x', 'mouse_4_ear_right_y', 'mouse_4_tail_base_x', 'mouse_4_tail_base_y', 'mouse_4_nose_x', 'mouse_4_nose_y', 'mouse_4_neck_x', 'mouse_4_neck_y', 'mouse_4_hip_left_x', 'mouse_4_hip_left_y', 'mouse_4_hip_right_x', 'mouse_4_hip_right_y', 'mouse1_nose_speed', 'mouse1_ear_left_speed', 'mouse1_ear_right_speed', 'mouse1_neck_speed', 'mouse1_tail_base_speed', 'mouse2_nose_speed', 'mouse2_ear_left_speed', 'mouse2_ear_right_speed', 'mouse2_neck_speed', 'mouse2_tail_base_speed', 'mouse3_nose_speed', 'mouse3_ear_left_speed', 'mouse3_ear_right_speed', 'mouse3_neck_speed', 'mouse3_tail_base_speed', 'mouse4_nose_speed', 'mouse4_ear_left_speed', 'mouse4_ear_right_speed', 'mouse4_neck_speed', 'mouse4_tail_base_speed', 'dist_m1nose_m2nose', 'dist_m1nose_m2ear_left', 'dist_m1nose_m2ear_right', 'dist_m1nose_m2neck', 'dist_m1nose_m2tail_base', 'dist_m1ear_left_m2nose', 'dist_m1ear_left_m2ear_left', 'dist_m1ear_left_m2ear_right', 'dist_m1ear_left_m2neck', 'dist_m1ear_left_m2tail_base', 'dist_m1ear_right_m2nose', 'dist_m1ear_right_m2ear_left', 'dist_m1ear_right_m2ear_right', 'dist_m1ear_right_m2neck', 'dist_m1ear_right_m2tail_base', 'dist_m1neck_m2nose', 'dist_m1neck_m2ear_left', 'dist_m1neck_m2ear_right', 'dist_m1neck_m2neck', 'dist_m1neck_m2tail_base', 'dist_m1tail_base_m2nose', 'dist_m1tail_base_m2ear_left', 'dist_m1tail_base_m2ear_right', 'dist_m1tail_base_m2neck', 'dist_m1tail_base_m2tail_base', 'dist_m1nose_m3nose', 'dist_m1nose_m3ear_left', 'dist_m1nose_m3ear_right', 'dist_m1nose_m3neck', 'dist_m1nose_m3tail_base', 'dist_m1ear_left_m3nose', 'dist_m1ear_left_m3ear_left', 'dist_m1ear_left_m3ear_right', 'dist_m1ear_left_m3neck', 'dist_m1ear_left_m3tail_base', 'dist_m1ear_right_m3nose', 'dist_m1ear_right_m3ear_left', 'dist_m1ear_right_m3ear_right', 'dist_m1ear_right_m3neck', 'dist_m1ear_right_m3tail_base', 'dist_m1neck_m3nose', 'dist_m1neck_m3ear_left', 'dist_m1neck_m3ear_right', 'dist_m1neck_m3neck', 'dist_m1neck_m3tail_base', 'dist_m1tail_base_m3nose', 'dist_m1tail_base_m3ear_left', 'dist_m1tail_base_m3ear_right', 'dist_m1tail_base_m3neck', 'dist_m1tail_base_m3tail_base', 'dist_m1nose_m4nose', 'dist_m1nose_m4ear_left', 'dist_m1nose_m4ear_right', 'dist_m1nose_m4neck', 'dist_m1nose_m4tail_base', 'dist_m1ear_left_m4nose', 'dist_m1ear_left_m4ear_left', 'dist_m1ear_left_m4ear_right', 'dist_m1ear_left_m4neck', 'dist_m1ear_left_m4tail_base', 'dist_m1ear_right_m4nose', 'dist_m1ear_right_m4ear_left', 'dist_m1ear_right_m4ear_right', 'dist_m1ear_right_m4neck', 'dist_m1ear_right_m4tail_base', 'dist_m1neck_m4nose', 'dist_m1neck_m4ear_left', 'dist_m1neck_m4ear_right', 'dist_m1neck_m4neck', 'dist_m1neck_m4tail_base', 'dist_m1tail_base_m4nose', 'dist_m1tail_base_m4ear_left', 'dist_m1tail_base_m4ear_right', 'dist_m1tail_base_m4neck', 'dist_m1tail_base_m4tail_base', 'dist_m2nose_m3nose', 'dist_m2nose_m3ear_left', 'dist_m2nose_m3ear_right', 'dist_m2nose_m3neck', 'dist_m2nose_m3tail_base', 'dist_m2ear_left_m3nose', 'dist_m2ear_left_m3ear_left', 'dist_m2ear_left_m3ear_right', 'dist_m2ear_left_m3neck', 'dist_m2ear_left_m3tail_base', 'dist_m2ear_right_m3nose', 'dist_m2ear_right_m3ear_left', 'dist_m2ear_right_m3ear_right', 'dist_m2ear_right_m3neck', 'dist_m2ear_right_m3tail_base', 'dist_m2neck_m3nose', 'dist_m2neck_m3ear_left', 'dist_m2neck_m3ear_right', 'dist_m2neck_m3neck', 'dist_m2neck_m3tail_base', 'dist_m2tail_base_m3nose', 'dist_m2tail_base_m3ear_left', 'dist_m2tail_base_m3ear_right', 'dist_m2tail_base_m3neck', 'dist_m2tail_base_m3tail_base', 'dist_m2nose_m4nose', 'dist_m2nose_m4ear_left', 'dist_m2nose_m4ear_right', 'dist_m2nose_m4neck', 'dist_m2nose_m4tail_base', 'dist_m2ear_left_m4nose', 'dist_m2ear_left_m4ear_left', 'dist_m2ear_left_m4ear_right', 'dist_m2ear_left_m4neck', 'dist_m2ear_left_m4tail_base', 'dist_m2ear_right_m4nose', 'dist_m2ear_right_m4ear_left', 'dist_m2ear_right_m4ear_right', 'dist_m2ear_right_m4neck', 'dist_m2ear_right_m4tail_base', 'dist_m2neck_m4nose', 'dist_m2neck_m4ear_left', 'dist_m2neck_m4ear_right', 'dist_m2neck_m4neck', 'dist_m2neck_m4tail_base', 'dist_m2tail_base_m4nose', 'dist_m2tail_base_m4ear_left', 'dist_m2tail_base_m4ear_right', 'dist_m2tail_base_m4neck', 'dist_m2tail_base_m4tail_base', 'dist_m3nose_m4nose', 'dist_m3nose_m4ear_left', 'dist_m3nose_m4ear_right', 'dist_m3nose_m4neck', 'dist_m3nose_m4tail_base', 'dist_m3ear_left_m4nose', 'dist_m3ear_left_m4ear_left', 'dist_m3ear_left_m4ear_right', 'dist_m3ear_left_m4neck', 'dist_m3ear_left_m4tail_base', 'dist_m3ear_right_m4nose', 'dist_m3ear_right_m4ear_left', 'dist_m3ear_right_m4ear_right', 'dist_m3ear_right_m4neck', 'dist_m3ear_right_m4tail_base', 'dist_m3neck_m4nose', 'dist_m3neck_m4ear_left', 'dist_m3neck_m4ear_right', 'dist_m3neck_m4neck', 'dist_m3neck_m4tail_base', 'dist_m3tail_base_m4nose', 'dist_m3tail_base_m4ear_left', 'dist_m3tail_base_m4ear_right', 'dist_m3tail_base_m4neck', 'dist_m3tail_base_m4tail_base']
print(len(final_cols))


['mouse_1_ear_left_x', 'mouse_1_ear_left_y', 'mouse_1_ear_right_x', 'mouse_1_ear_right_y', 'mouse_1_tail_base_x', 'mouse_1_tail_base_y', 'mouse_1_nose_x', 'mouse_1_nose_y', 'mouse_1_neck_x', 'mouse_1_neck_y', 'mouse_1_hip_left_x', 'mouse_1_hip_left_y', 'mouse_1_hip_right_x', 'mouse_1_hip_right_y', 'mouse_2_ear_left_x', 'mouse_2_ear_left_y', 'mouse_2_ear_right_x', 'mouse_2_ear_right_y', 'mouse_2_tail_base_x', 'mouse_2_tail_base_y', 'mouse_2_nose_x', 'mouse_2_nose_y', 'mouse_2_neck_x', 'mouse_2_neck_y', 'mouse_2_hip_left_x', 'mouse_2_hip_left_y', 'mouse_2_hip_right_x', 'mouse_2_hip_right_y', 'mouse_3_ear_left_x', 'mouse_3_ear_left_y', 'mouse_3_ear_right_x', 'mouse_3_ear_right_y', 'mouse_3_tail_base_x', 'mouse_3_tail_base_y', 'mouse_3_nose_x', 'mouse_3_nose_y', 'mouse_3_neck_x', 'mouse_3_neck_y', 'mouse_3_hip_left_x', 'mouse_3_hip_left_y', 'mouse_3_hip_right_x', 'mouse_3_hip_right_y', 'mouse_4_ear_left_x', 'mouse_4_ear_left_y', 'mouse_4_ear_right_x', 'mouse_4_ear_right_y', 'mouse_4_tail_b

In [4]:
# List of mouse IDs and core body parts
mouse_ids = [1, 2, 3, 4]
CORE_BODYPARTS = ['nose', 'ear_left', 'ear_right', 'neck', 'tail_base']
# Start grouping columns by mouse
mouse_columns = {mid: [] for mid in mouse_ids}
interaction_columns = []

for col in final_cols:
    added = False
    for mid in mouse_ids:
        if f'mouse_{mid}_' in col or f'mouse{mid}_' in col:
            mouse_columns[mid].append(col)
            added = True
            break
    if not added and col.startswith('dist_m'):
        interaction_columns.append(col)  # These are pairwise interaction features

# Reconstruct column order: mouse 1 features → mouse 2 features → ... → interaction features
new_column_order = []
for mid in mouse_ids:
    new_column_order.extend(sorted(mouse_columns[mid]))  # Optionally sort each group
new_column_order.extend(sorted(interaction_columns))  # Sort interaction features last
print(new_column_order)
print(len(interaction_columns))

['mouse1_ear_left_speed', 'mouse1_ear_right_speed', 'mouse1_neck_speed', 'mouse1_nose_speed', 'mouse1_tail_base_speed', 'mouse_1_ear_left_x', 'mouse_1_ear_left_y', 'mouse_1_ear_right_x', 'mouse_1_ear_right_y', 'mouse_1_hip_left_x', 'mouse_1_hip_left_y', 'mouse_1_hip_right_x', 'mouse_1_hip_right_y', 'mouse_1_neck_x', 'mouse_1_neck_y', 'mouse_1_nose_x', 'mouse_1_nose_y', 'mouse_1_tail_base_x', 'mouse_1_tail_base_y', 'mouse2_ear_left_speed', 'mouse2_ear_right_speed', 'mouse2_neck_speed', 'mouse2_nose_speed', 'mouse2_tail_base_speed', 'mouse_2_ear_left_x', 'mouse_2_ear_left_y', 'mouse_2_ear_right_x', 'mouse_2_ear_right_y', 'mouse_2_hip_left_x', 'mouse_2_hip_left_y', 'mouse_2_hip_right_x', 'mouse_2_hip_right_y', 'mouse_2_neck_x', 'mouse_2_neck_y', 'mouse_2_nose_x', 'mouse_2_nose_y', 'mouse_2_tail_base_x', 'mouse_2_tail_base_y', 'mouse3_ear_left_speed', 'mouse3_ear_right_speed', 'mouse3_neck_speed', 'mouse3_nose_speed', 'mouse3_tail_base_speed', 'mouse_3_ear_left_x', 'mouse_3_ear_left_y', 'm

In [5]:
def rotate_keypoints(df: pd.DataFrame, width: int, height: int, angle: int) -> pd.DataFrame:
    if angle == 0:
        return df
    if angle not in [90, 180, 270]:
        raise ValueError("Angle must be one of: 90, 180, or 270 degrees")

    df_rot = df.copy()

    # Get all unique keypoint prefixes (like 'mouse_1_ear_left', 'mouse_2_nose', etc.)
    base_names = sorted({col.rsplit('_', 1)[0] for col in df.columns})

    for name in base_names:
        x_col = f"{name}_x"
        y_col = f"{name}_y"

        x = df[x_col]
        y = df[y_col]

        if angle == 90:
            x_new = height - y - 1
            y_new = x
        elif angle == 180:
            x_new = width - x - 1
            y_new = height - y - 1
        elif angle == 270:
            x_new = y
            y_new = width - x - 1

        df_rot[x_col] = x_new
        df_rot[y_col] = y_new

    return df_rot

In [6]:
def normalize(data, FRAME_WIDTH_TOP, FRAME_HEIGHT_TOP):
    """Normalize coordinate data (NumPy array or DataFrame)."""
    is_df = isinstance(data, pd.DataFrame)
    values = data.values if is_df else data
    if values.shape[1] % 2 != 0:
        raise ValueError("Expected even number of columns representing (x, y) coordinate pairs.")

    state_dim = values.shape[1] // 2
    shift = np.array([FRAME_WIDTH_TOP / 2, FRAME_HEIGHT_TOP / 2] * state_dim)
    scale = np.array([FRAME_WIDTH_TOP / 2, FRAME_HEIGHT_TOP / 2] * state_dim)

    normalized = (values - shift) / scale
    # normalized = np.where(np.isnan(values), SPECIAL_VALUE, normalized)

    if is_df:
        return pd.DataFrame(normalized, columns=data.columns, index=data.index)
    else:
        return normalized

def unnormalize(data, FRAME_WIDTH_TOP, FRAME_HEIGHT_TOP):
    """
    Undo normalization of coordinate data.
    Accepts NumPy arrays, Pandas DataFrames, or PyTorch tensors.
    Expects data in the format: [batch_size, x1, y1, x2, y2, ..., xn, yn]
    """

    # Convert torch tensor to NumPy array
    if torch.is_tensor(data):
        data = data.detach().cpu().numpy()

    is_df = isinstance(data, pd.DataFrame)
    values = data.values if is_df else data

    state_dim = values.shape[1] // 2

    x_shift = FRAME_WIDTH_TOP / 2
    y_shift = FRAME_HEIGHT_TOP / 2
    x_scale = FRAME_WIDTH_TOP / 2
    y_scale = FRAME_HEIGHT_TOP / 2

    # Unnormalize x and y coordinates separately
    values[:, ::2] = values[:, ::2] * x_scale + x_shift  # x coordinates
    values[:, 1::2] = values[:, 1::2] * y_scale + y_shift  # y coordinates

    # Return in original format
    if is_df:
        return pd.DataFrame(values, columns=data.columns, index=data.index)
    else:
        return values  # shape remains [batch_size, state_dim * 2]


def rotate(data, center_index):
    # data shape is num_seq x 3 x 10 x 2
    
    data = data.reshape(data.shape[0], 4, 29 ,2)
    mice = [data[:,i,:10,:] for i in range(3)]
    data = np.concatenate(mice, axis=0)

    del mice
    gc.collect()

    mouse_center = data[:, center_index, :]
    centered_data = data - mouse_center[:, np.newaxis, :]

	# Rotate such that keypoints 3 and 6 are parallel with the y axis
    mouse_rotation = np.arctan2(
		data[:, 3, 0] - data[:, 9, 0], data[:, 3, 1] - data[:, 9, 1])

    R = (np.array([[np.cos(mouse_rotation), -np.sin(mouse_rotation)],
				   [np.sin(mouse_rotation),  np.cos(mouse_rotation)]]).transpose((2, 0, 1)))

	# Encode mouse rotation as sine and cosine
    mouse_rotation = np.concatenate([np.sin(mouse_rotation)[:, np.newaxis], np.cos(
		mouse_rotation)[:, np.newaxis]], axis=-1)

    centered_data = np.matmul(R, centered_data.transpose(0, 2, 1))
    centered_data = centered_data.transpose((0, 2, 1))
    centered_data = centered_data.reshape((-1, 20))

    return centered_data, mouse_center, mouse_rotation

In [7]:
def load_and_process_video(video_id, lab_id, start, end, width, height, angle, data_path=DATA_PATH):
    # print(f"doing from {start} to {end}")
    tracking_path = os.path.join(data_path, 'train_tracking', lab_id, f'{video_id}.parquet')
    if not os.path.exists(tracking_path):
        return None
    df = reassign_points(tracking_path, start, end)
    # df.set_index(['video_frame', 'mouse_id'], inplace=True)

    # # Step 2: Flatten the structure by pivoting mouse_id into columns
    # df_wide = df.unstack(level='mouse_id')

    # # Step 3: Flatten the MultiIndex column names
    # df_wide.columns = [f'mouse_{mouse_id}_{col}' for col, mouse_id in df_wide.columns]

    df_wide = df.reindex(columns=all_cols)
    df_wide = rotate_keypoints(df_wide, width, height, angle)
    if angle in [90, 270]:
        df_wide = normalize(df_wide, height, width)
    else:
        df_wide = normalize(df_wide, width, height)
    # df_wide = unnormalize(df_wide, width, height)
    df_wide = create_hybrid_features(df_wide)
    df_wide = df_wide.reindex(columns=new_column_order)
    # df_wide.to_csv('output.csv', index=False)

    # For debugging
    # print(f"df long cols {list(df_wide.columns)}")
    # print(f"df long {df_wide.head}")
    # print(f"df wide shape {df_wide.shape}")
    return df_wide

CORE_BODYPARTS = ['nose', 'ear_left', 'ear_right', 'neck' 'tail_base']
def create_advanced_features(df_wide):
    """
    Creates a rich set of kinematic, interaction, and postural features.
    """
    # Start with a copy of the original data
    features_df = df_wide.copy()
    
    mouse_ids = [1, 2, 3, 4] # Assuming up to 4 mice
    
    # --- 1. Kinematic Features (Speeds) ---
    for mid in mouse_ids:
        for part in CORE_BODYPARTS:
            col_x, col_y = f'mouse_{mid}_{part}_x', f'mouse_{mid}_{part}_y'
            if col_x in features_df.columns:
                delta_x = features_df[col_x].diff()
                delta_y = features_df[col_y].diff()
                features_df[f'mouse{mid}_{part}_speed'] = np.sqrt(delta_x**2 + delta_y**2)

    # --- 2. Postural Features (Body Elongation) ---
    for mid in mouse_ids:
        nose_x, nose_y = f'mouse_{mid}_top_center_x', f'mouse_{mid}_top_center_y'
        tail_x, tail_y = f'mouse_{mid}_bottom_x', f'mouse_{mid}_bottom_y'
        if all(c in features_df.columns for c in [nose_x, nose_y, tail_x, tail_y]):
            features_df[f'mouse{mid}_elongation'] = np.sqrt(
                (features_df[nose_x] - features_df[tail_x])**2 + 
                (features_df[nose_y] - features_df[tail_y])**2
            )

    # --- 3. Interaction Features (Distances) ---
    mouse_pairs = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
    for m1, m2 in mouse_pairs:
        for part1 in CORE_BODYPARTS:
            for part2 in CORE_BODYPARTS:
                p1_x, p1_y = f'mouse_{m1}_{part1}_x', f'mouse_{m1}_{part1}_y'
                p2_x, p2_y = f'mouse_{m2}_{part2}_x', f'mouse_{m2}_{part2}_y'
                if all(c in features_df.columns for c in [p1_x, p1_y, p2_x, p2_y]):
                    features_df[f'dist_m{m1}{part1}_m{m2}{part2}'] = np.sqrt(
                        (features_df[p1_x] - features_df[p2_x])**2 + 
                        (features_df[p1_y] - features_df[p2_y])**2
                    )
                    
    # Drop the original coordinate columns to force the model to use our new features
    features_df = features_df.copy()
    features_df = features_df.drop(columns=df_wide.columns)
    
    return features_df

def create_hybrid_features(df_wide):
    engineered_features = create_advanced_features(df_wide.copy())
    
    hybrid_features = pd.concat([df_wide, engineered_features], axis=1)
    return hybrid_features

In [8]:
dir_path = os.path.join(DATA_PATH, "train_annotation")

# Recursively find all .parquet files
ann_files = []
for root, _, files in os.walk(dir_path):
    for file in files:
        if file.endswith(".parquet"):
            ann_files.append(os.path.join(root, file))

# Read and concatenate all parquet files into one DataFrame
dfs = []
for f in ann_files:
    df = pd.read_parquet(f)
    fn = os.path.splitext(os.path.basename(f))[0]  # filename without extension
    df['video_file'] = fn
    dfs.append(df)

train_ann = pd.concat(dfs, ignore_index=True)

print(f"Number of files: {len(ann_files)}")
# print(train_ann.head(10))
self_actions = train_ann.loc[train_ann['agent_id'] == train_ann['target_id'], 'action'].unique()

# print("Self actions:", ", ".join(self_actions))

# Pair actions: where agent_id != target_id, unique actions
pair_actions = train_ann.loc[train_ann['agent_id'] != train_ann['target_id'], 'action'].unique()

# print("\nPair actions:", ", ".join(pair_actions))
all_actions = np.unique(np.concatenate((self_actions, pair_actions)))
all_actions = list(all_actions)
self_actions = list(self_actions)
pair_actions = list(pair_actions)
all_actions.append("no_behavior")
self_actions.insert(0, "no_behavior")
pair_actions.insert(0, "no_behavior")

solo_encoder = LabelEncoder()
solo_encoder.classes_ = np.array(self_actions)
dual_encoder = LabelEncoder()
dual_encoder.classes_ = np.array(pair_actions)
del train_ann, ann_files

agents = [1, 2, 3, 4]
agent_encoder = LabelEncoder()
agent_encoder.fit(np.array(agents).reshape(-1, 1))
print(len(self_actions), len(pair_actions))

Number of files: 847
12 27


c:\Users\PC\.conda\envs\reg\lib\site-packages\sklearn\preprocessing\_label.py:93: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [9]:
def build_chunks_test(path, train_df_path, all_files, chunk_size=CHUNK_SIZE, sub_seq_length=SUB_SEQ_LENGTH):
    chunks = []
    print(f"building test chunks with {len(all_files)} files")

    train_df = pd.read_csv(train_df_path)
    for _, row in train_df.iterrows():
        video_path = os.path.join(path, 'train_annotation', row['lab_id'], f"{row['video_id']}.parquet")
        if video_path not in all_files:
            continue

        df = pd.read_parquet(video_path)

        start_frame = df['start_frame'].iloc[0]
        stop_frame = df['stop_frame'].iloc[-1]

        for start_idx in range(start_frame, stop_frame + 1, chunk_size):
            start_frame2 = max(start_frame, start_idx - sub_seq_length // 2)
            chunk_end = min(start_idx + chunk_size + sub_seq_length // 2, stop_frame)
            chunks.append({
                'video_id': row['video_id'],
                'lab_id': str(row['lab_id']),
                'width': row['video_width_pix'],
                'height': row['video_height_pix'],
                'chunk_start': start_frame2,
                'chunk_end': chunk_end,
                'first_chunk': start_idx == start_frame,
                'last_chunk': chunk_end == stop_frame,
                'angle': 0
            })
    return chunks

def build_chunks(path, train_df_path, all_files, sub_seq_length=SUB_SEQ_LENGTH, min_chunk_len=22, max_chunk_len=2500):
    chunks = []
    print(f"Building chunks with {len(all_files)} files")

    train_df = pd.read_csv(train_df_path)

    for _, row in train_df.iterrows():
        video_path = os.path.join(path, 'train_annotation', str(row['lab_id']), f"{row['video_id']}.parquet")
        if video_path not in all_files:
            continue

        df = pd.read_parquet(video_path)
        df = df.sort_values('start_frame').reset_index(drop=True)

        final_frame = int(df['stop_frame'].iloc[-1])

        # --- 1️⃣ Behavior chunks ---
        for _, ann in df.iterrows():
            start_frame = int(ann['start_frame'])
            stop_frame = int(ann['stop_frame'])
            length = stop_frame - start_frame + 1

            if length < min_chunk_len or length > max_chunk_len:
                continue

            chunks.append({
                'video_id': row['video_id'],
                'lab_id': str(row['lab_id']),
                'width': row['video_width_pix'],
                'height': row['video_height_pix'],
                'chunk_start': start_frame,
                'chunk_end': stop_frame,
                'label': ann['action'],
                'agent_id': ann['agent_id'],
                'target_id': ann['target_id'],
                'first_chunk': start_frame < 10,
                'last_chunk': stop_frame == final_frame,
                'angle': 0,
            })

        # --- 2️⃣ No-behavior chunks ---
        # prev_stop = 0
        # for _, ann in df.iterrows():
        #     start_frame = int(ann['start_frame'])
        #     if start_frame - prev_stop - 1 >= min_chunk_len:
        #         chunks.append({
        #             'video_id': row['video_id'],
        #             'lab_id': str(row['lab_id']),
        #             'width': row['video_width_pix'],
        #             'height': row['video_height_pix'],
        #             'chunk_start': prev_stop + 1,
        #             'chunk_end': start_frame - 1,
        #             'label': 'no_behavior',
        #             'agent_id': None,
        #             'target_id': None,
        #             'first_chunk': prev_stop < 10,
        #             'last_chunk': False,
        #             'angle': 0,
        #         })
        #     prev_stop = int(ann['stop_frame'])

        # # --- 3️⃣ Tail-end no-behavior chunk ---
        # if final_frame - prev_stop >= min_chunk_len:
        #     chunks.append({
        #         'video_id': row['video_id'],
        #         'lab_id': str(row['lab_id']),
        #         'width': row['video_width_pix'],
        #         'height': row['video_height_pix'],
        #         'chunk_start': prev_stop + 1,
        #         'chunk_end': final_frame,
        #         'label': 'no_behavior',
        #         'agent_id': None,
        #         'target_id': None,
        #         'first_chunk': False,
        #         'last_chunk': True,
        #         'angle': 0,
        #     })

    return chunks

In [10]:
def split_files_into_train_test(path, train_df_path):
    all_files = []
    train_df = pd.read_csv(train_df_path)
    final_files = []

    # Gather all file paths
    for dirpath, _, filenames in os.walk(path):
        for filename in filenames:
            all_files.append(os.path.join(dirpath, filename))

    for _, row in train_df.iterrows():
        video_path = os.path.join(path, 'train_annotation', str(row['lab_id']), f"{row['video_id']}.parquet")
        if video_path not in all_files:
            continue
        else:
            final_files.append(video_path)


    train_files, test_files = train_test_split(final_files, test_size=0.1, random_state=42)

    return train_files, test_files

In [11]:
solo_encoder.classes_

array(['no_behavior', 'rear', 'genitalgroom', 'selfgroom', 'dig', 'run',
       'rest', 'climb', 'freeze', 'exploreobject', 'biteobject', 'huddle'],
      dtype='<U13')

In [12]:
class MABEModelDataset(Dataset):
    def __init__(self, path, solo_encoder, dual_encoder, chunks):
        self.agent_encoder = agent_encoder
        self.path = path
        self.solo_encoder = solo_encoder
        self.dual_encoder = dual_encoder
        self.chunk_size = CHUNK_SIZE
        self.sub_seq_length = SUB_SEQ_LENGTH
        self.sliding_window = SLIDING_WINDOW
        self.all_files = []

        self.chunks = chunks 
        print(f"num chunks: {len(self.chunks)}")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        try:
            info = self.chunks[idx]
            video_id = info['video_id']
            lab_id = info['lab_id']
            width = info['width']
            height = info['height']
            chunk_start = info['chunk_start']
            chunk_end = info['chunk_end']
            is_start = info['first_chunk']
            is_end = info['last_chunk']
            angle = info['angle']
            # print(f"for video {video_id} {lab_id} {chunk_start} {chunk_end}")
        
            half_window = self.sub_seq_length // 2  # 10 for length 21
        
            # Load annotations once to know full video length
            ann_df = pd.read_parquet(os.path.join(self.path, 'train_annotation', lab_id, f"{video_id}.parquet"))
            total_video_length = ann_df['stop_frame'].max() + 1
        
            # --- Adjust chunk boundaries for context frames ---
            load_start = chunk_start
            load_end = chunk_end
            if not is_start:
                load_start = max(0, chunk_start - half_window)
            if not is_end:
                load_end = min(total_video_length - 1, chunk_end + half_window)
        
            # --- Load frames with context ---
            df = load_and_process_video(video_id, lab_id, load_start, load_end, width, height, angle=angle)
            vec_seq = df.to_numpy()
        
            # --- Pad edges only if we’re at start or end ---
            if is_start:
                left_pad = np.repeat(vec_seq[0:1], half_window, axis=0)
                vec_seq = np.concatenate([left_pad, vec_seq], axis=0)
        
            if is_end:
                right_pad = np.repeat(vec_seq[-1:], half_window, axis=0)
                vec_seq = np.concatenate([vec_seq, right_pad], axis=0)
        
            # --- Recompute true valid range within vec_seq ---
            start_offset = 0 if is_start else half_window
            end_offset = vec_seq.shape[0] - half_window if is_end else vec_seq.shape[0] - half_window
        
            # --- Build centered subsequences (length = 21) ---
            sub_seqs = np.stack([
                vec_seq[i - half_window:i + half_window + 1]
                for i in range(half_window, vec_seq.shape[0] - half_window)
            ])  # shape: [num_frames_in_chunk, 21, state_dim]
        
            original_len = sub_seqs.shape[0]
        
            # --- Build frame-level labels for the original (unpadded) chunk ---
            frame_labels = {i: np.full(original_len, 'no_behavior', dtype=object) for i in range(1, 5)}
            frame_targets = {i: np.full(original_len, 0, dtype=int) for i in range(1, 5)}
        
            # Filter annotations that overlap with current chunk
            for _, row in ann_df.iterrows():
                start, stop, action, agent, target = row['start_frame'], row['stop_frame'], row['action'], row['agent_id'], row['target_id']
        
                # Skip if outside current chunk boundaries
                if stop < chunk_start or start > chunk_end:
                    continue
        
                # Convert to local frame indices within current chunk
                local_start = max(0, start - chunk_start)
                local_stop = min(original_len - 1, stop - chunk_start)
        
                frame_labels[agent][local_start:local_stop + 1] = action
                frame_targets[agent][local_start:local_stop + 1] = target
        
            # --- Extract labels for the center frame (index 10) of each subsequence ---
            mouse_labels = {i: [] for i in range(1, 5)}
            mouse_targets = {i: [] for i in range(1, 5)}
        
            for i in range(original_len):
                center_frame = i + chunk_start  # actual frame index in video
                for mouse_id in range(1, 5):
                    label = frame_labels[mouse_id][i]
                    target = frame_targets[mouse_id][i]
                    mouse_labels[mouse_id].append(label)
                    mouse_targets[mouse_id].append(target)
        
            num_agents = 4
            num_solo_classes = len(self.solo_encoder.classes_)
            num_dual_classes = len(self.dual_encoder.classes_)

            # Shapes:
            # solo_labels: [T, 4]  (long ints: class indices for solo encoder)
            # dual_labels: [T, 4, 4] (long ints: class indices for dual encoder; diagonal stays 0)
            solo_labels = torch.zeros((original_len, num_agents), dtype=torch.long)
            dual_labels = torch.zeros((original_len, num_agents, num_agents), dtype=torch.long)

            for f in range(original_len):
                for agent in range(1, num_agents + 1):
                    action_str = frame_labels[agent][f]  # e.g. 'attack' or 'no_behavior'

                    # --- SOLO: encode in separate solo_labels matrix ---
                    if action_str in self_actions:
                        solo_idx = self.solo_encoder.transform([action_str])[0]
                        solo_labels[f, agent - 1] = solo_idx

                    # --- DUAL: encode only non-self, non-zero targets ---
                    target = frame_targets[agent][f]  # 0 means no target
                    if target != 0 and target != agent:
                        target_idx = self.agent_encoder.transform([target])[0]  # e.g. 0..3
                        dual_idx = self.dual_encoder.transform([action_str])[0]
                        dual_labels[f, agent - 1, target_idx] = dual_idx
            states = sub_seqs
        
            # --- Sanity check ---
            # print(f"Chunk [{chunk_start}, {chunk_end}] | vec_seq: {vec_seq.shape} | subseqs: {sub_seqs.shape}")
        
            # for i in range(min(20, solo_labels.shape[0])):  # print first few
            #     # for j in range(1, 5):
            #     j = 2
            #     print(f"Frame {i + chunk_start} | dual labels: {dual_labels[i]}")
            # print(f"states shape {states.shape}")

            mouse_states = {}
            for i in range(4):
                start_col = i * 19
                end_col = (i + 1) * 19
                mouse_states[i + 1] = states[:,:, start_col:end_col] 
            dist_feats = states[:, :, 76:]

            has_action = torch.zeros((original_len, 1), dtype=torch.long)

            for f in range(original_len):
                solo_nonzero = (solo_labels[f] != 0).any()  # if any agent has a solo action
                dual_nonzero = (dual_labels[f] != 0).any()  # if any agent-target pair has a dual action
                if solo_nonzero or dual_nonzero:
                    has_action[f, 0] = 1
        
            return (
                mouse_states[1],
                mouse_states[2],
                mouse_states[3],
                mouse_states[4],
                dist_feats,
                solo_labels,
                dual_labels,
                has_action,
                # video_id,
                # chunk_start
            )
        except:
            print("returning None")
            return None
    
class MABEModelDatasetEval(Dataset):
    def __init__(self, path, solo_encoder, dual_encoder, chunks):
        self.agent_encoder = agent_encoder
        self.path = path
        self.solo_encoder = solo_encoder
        self.dual_encoder = dual_encoder
        self.chunk_size = CHUNK_SIZE
        self.sub_seq_length = SUB_SEQ_LENGTH
        self.sliding_window = SLIDING_WINDOW
        self.all_files = []

        self.chunks = chunks 
        print(f"num chunks: {len(self.chunks)}")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        try:
            info = self.chunks[idx]
            video_id = info['video_id']
            lab_id = info['lab_id']
            width = info['width']
            height = info['height']
            chunk_start = info['chunk_start']
            chunk_end = info['chunk_end']
            is_start = info['first_chunk']
            is_end = info['last_chunk']
            angle = info['angle']
            # print(f"for video {video_id} {lab_id} {chunk_start} {chunk_end}")
        
            half_window = self.sub_seq_length // 2  # 10 for length 21
        
            # Load annotations once to know full video length
            ann_df = pd.read_parquet(os.path.join(self.path, 'train_annotation', lab_id, f"{video_id}.parquet"))
            total_video_length = ann_df['stop_frame'].max() + 1
        
            # --- Adjust chunk boundaries for context frames ---
            load_start = chunk_start
            load_end = chunk_end
            if not is_start:
                load_start = max(0, chunk_start - half_window)
            if not is_end:
                load_end = min(total_video_length - 1, chunk_end + half_window)
        
            # --- Load frames with context ---
            df = load_and_process_video(video_id, lab_id, load_start, load_end, width, height, angle=angle)
            vec_seq = df.to_numpy()
        
            # --- Pad edges only if we’re at start or end ---
            if is_start:
                left_pad = np.repeat(vec_seq[0:1], half_window, axis=0)
                vec_seq = np.concatenate([left_pad, vec_seq], axis=0)
        
            if is_end:
                right_pad = np.repeat(vec_seq[-1:], half_window, axis=0)
                vec_seq = np.concatenate([vec_seq, right_pad], axis=0)
        
            # --- Recompute true valid range within vec_seq ---
            start_offset = 0 if is_start else half_window
            end_offset = vec_seq.shape[0] - half_window if is_end else vec_seq.shape[0] - half_window
        
            # --- Build centered subsequences (length = 21) ---
            sub_seqs = np.stack([
                vec_seq[i - half_window:i + half_window + 1]
                for i in range(half_window, vec_seq.shape[0] - half_window)
            ])  # shape: [num_frames_in_chunk, 21, state_dim]
        
            original_len = sub_seqs.shape[0]
        
            # --- Build frame-level labels for the original (unpadded) chunk ---
            frame_labels = {i: np.full(original_len, 'no_behavior', dtype=object) for i in range(1, 5)}
            frame_targets = {i: np.full(original_len, 0, dtype=int) for i in range(1, 5)}
        
            # Filter annotations that overlap with current chunk
            for _, row in ann_df.iterrows():
                start, stop, action, agent, target = row['start_frame'], row['stop_frame'], row['action'], row['agent_id'], row['target_id']
        
                # Skip if outside current chunk boundaries
                if stop < chunk_start or start > chunk_end:
                    continue
        
                # Convert to local frame indices within current chunk
                local_start = max(0, start - chunk_start)
                local_stop = min(original_len - 1, stop - chunk_start)
        
                frame_labels[agent][local_start:local_stop + 1] = action
                frame_targets[agent][local_start:local_stop + 1] = target
        
            # --- Extract labels for the center frame (index 10) of each subsequence ---
            mouse_labels = {i: [] for i in range(1, 5)}
            mouse_targets = {i: [] for i in range(1, 5)}
        
            for i in range(original_len):
                center_frame = i + chunk_start  # actual frame index in video
                for mouse_id in range(1, 5):
                    label = frame_labels[mouse_id][i]
                    target = frame_targets[mouse_id][i]
                    mouse_labels[mouse_id].append(label)
                    mouse_targets[mouse_id].append(target)
        
            num_agents = 4
            num_solo_classes = len(self.solo_encoder.classes_)
            num_dual_classes = len(self.dual_encoder.classes_)

            # Shapes:
            # solo_labels: [T, 4]  (long ints: class indices for solo encoder)
            # dual_labels: [T, 4, 4] (long ints: class indices for dual encoder; diagonal stays 0)
            solo_labels = torch.zeros((original_len, num_agents), dtype=torch.long)
            dual_labels = torch.zeros((original_len, num_agents, num_agents), dtype=torch.long)

            for f in range(original_len):
                for agent in range(1, num_agents + 1):
                    action_str = frame_labels[agent][f]  # e.g. 'attack' or 'no_behavior'

                    # --- SOLO: encode in separate solo_labels matrix ---
                    if action_str in self_actions:
                        solo_idx = self.solo_encoder.transform([action_str])[0]
                        solo_labels[f, agent - 1] = solo_idx

                    # --- DUAL: encode only non-self, non-zero targets ---
                    target = frame_targets[agent][f]  # 0 means no target
                    if target != 0 and target != agent:
                        target_idx = self.agent_encoder.transform([target])[0]  # e.g. 0..3
                        dual_idx = self.dual_encoder.transform([action_str])[0]
                        dual_labels[f, agent - 1, target_idx] = dual_idx
            states = sub_seqs
        
            # --- Sanity check ---
            # print(f"Chunk [{chunk_start}, {chunk_end}] | vec_seq: {vec_seq.shape} | subseqs: {sub_seqs.shape}")
        
            # for i in range(min(20, solo_labels.shape[0])):  # print first few
            #     # for j in range(1, 5):
            #     j = 2
            #     print(f"Frame {i + chunk_start} | dual labels: {dual_labels[i]}")
            # print(f"states shape {states.shape}")

            mouse_states = {}
            for i in range(4):
                start_col = i * 19
                end_col = (i + 1) * 19
                mouse_states[i + 1] = states[:,:, start_col:end_col] 
            dist_feats = states[:, :, 76:]

            has_action = torch.zeros((original_len, 1), dtype=torch.long)

            for f in range(original_len):
                solo_nonzero = (solo_labels[f] != 0).any()  # if any agent has a solo action
                dual_nonzero = (dual_labels[f] != 0).any()  # if any agent-target pair has a dual action
                if solo_nonzero or dual_nonzero:
                    has_action[f, 0] = 1
        
            return (
                mouse_states[1],
                mouse_states[2],
                mouse_states[3],
                mouse_states[4],
                dist_feats,
                solo_labels,
                dual_labels,
                has_action,
                video_id,
                chunk_start
            )
        except:
            print("returning None")
            return None

In [13]:
def filter_chunks_by_inactive_frames(chunks, inactive_threshold=CHUNK_SIZE):
    filtered_chunks = []

    for info in chunks:
        lab_id = info['lab_id']
        video_id = info['video_id']
        chunk_start = info['chunk_start']
        chunk_end = info['chunk_end']

        annotation_path = os.path.join(DATA_PATH, 'train_annotation', lab_id, f"{video_id}.parquet")
        if not os.path.exists(annotation_path):
            print(f"Warning: Annotation file not found for {video_id} in {lab_id}")
            continue

        df_annotations = pd.read_parquet(annotation_path)

        # Frames in chunk
        chunk_frames = set(range(chunk_start, chunk_end + 1))

        # Filter annotations that overlap with chunk
        overlapping = df_annotations[
            (df_annotations['stop_frame'] >= chunk_start) &
            (df_annotations['start_frame'] <= chunk_end)
        ]

        # Collect active frames
        active_frames = set()
        for _, row in overlapping.iterrows():
            start = max(row['start_frame'], chunk_start)
            stop = min(row['stop_frame'], chunk_end)
            active_frames.update(range(start, stop + 1))

        # Determine inactive frames
        inactive_frames = chunk_frames - active_frames

        if len(inactive_frames) <= inactive_threshold:
            info['inactive_frames'] = len(inactive_frames)
            filtered_chunks.append(info)

    return filtered_chunks

# chunks = filter_chunks_by_inactive_frames(chunks, inactive_threshold=100)
# print(len(chunks))
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)


In [14]:
# train_files, test_files = split_files_into_train_test(DATA_PATH, os.path.join(DATA_PATH, 'train.csv'))
# train_chunks = build_chunks(DATA_PATH, os.path.join(DATA_PATH, "train.csv"), train_files)
# train_chunks = filter_chunks_by_inactive_frames(train_chunks, 100)
# chunk = train_chunks[10]
# new_chunk = chunk.copy()
# new_chunk['angle'] = 180
# new_chunks = [chunk, new_chunk]
# dataset = MABEModelDataset(DATA_PATH, solo_encoder, dual_encoder, new_chunks)
# i = 1
# all_batches = []
# for batch in dataset:
#     a, b, c, d, e, f, g, h = batch
#     all_batches.append(batch)
#     # if i % 1 == 0:
#     #     break
#     # else:
#     #     i += 1
# print(a.shape)
# print(d.shape)
# print(e.shape)
# print(h.shape)

In [15]:
# all_batches[0][4][10][0]

In [16]:
class CNN_BiLSTM_Model(nn.Module):
    def __init__(self, feat_dim, cnn_out_channels=128, lstm_hidden_size=128, use_mask=True):
        super(CNN_BiLSTM_Model, self).__init__()

        self.use_mask = use_mask
        input_channels = feat_dim + (1 if use_mask else 0)  # add mask channel if used

        # CNN expects: (batch, channels, length)
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_channels, out_channels=cnn_out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=cnn_out_channels, out_channels=cnn_out_channels, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # BiLSTM expects: (batch, seq_len, input_size)
        self.bilstm = nn.LSTM(
            input_size=cnn_out_channels,
            hidden_size=lstm_hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        # Final projection
        self.output_proj = nn.Linear(2 * lstm_hidden_size, cnn_out_channels)

    def forward(self, x, mask=None):
        """
        Args:
            x: Tensor of shape [B, T, F]
            mask: Tensor of shape [B, T, F] with 1 where feature is defined, 0 where missing
        Returns:
            out: [B, cnn_out_channels]
        """
        # print(f"input to cnn {x.shape} {mask.shape}")
        if mask is None:
            # assume all features are present
            mask = torch.ones_like(x)

        # print(x.shape)
        # Replace missing values with 0
        x = torch.where(mask.bool(), x, torch.zeros_like(x))

        if self.use_mask:
            # Collapse feature-level mask -> single validity per timestep
            # e.g. if any feature is valid, timestep is valid
            mask_channel = mask.mean(dim=-1, keepdim=True)  # [B, T, 1]
            x = torch.cat([x, mask_channel], dim=-1)  # [B, T, F+1]

        # Permute for CNN: (batch, channels, length)
        x = x.permute(0, 2, 1)  # [B, C, T]

        # CNN feature extraction
        x = self.cnn(x)  # [B, C_out, T]

        # Permute back for LSTM: (batch, T, C_out)
        x = x.permute(0, 2, 1)

        # Optional masking before pooling: ignore missing timesteps
        # (use mean pooling weighted by valid entries)
        lstm_out, _ = self.bilstm(x)  # [B, T, 2 * H]

        # Weighted mean pooling using mask (to ignore missing timesteps)
        # Collapse feature mask into [B, T, 1]
        time_mask = (mask.sum(dim=-1, keepdim=True) > 0).float()  # [B, T, 1]
        valid_counts = torch.clamp(time_mask.sum(dim=1), min=1.0)  # [B, 1]
        pooled = (lstm_out * time_mask).sum(dim=1) / valid_counts  # [B, 256]

        # Final projection
        out = self.output_proj(pooled)  # [B, cnn_out_channels]
        return out

In [17]:
class TransformerRelationalModel(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, hidden_dim=128,
                 num_solo_classes=12, num_dual_classes=27, max_agents=4, dropout=0.1):
        super(TransformerRelationalModel, self).__init__()

        self.embed_dim = embed_dim
        self.max_agents = max_agents

        # --- Embeddings ---
        self.agent_id_embed = nn.Embedding(max_agents, embed_dim)

        # --- Transformer Encoder (single layer) ---
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # --- SOLO Action Head ---
        self.solo_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_solo_classes)
        )

        # --- DUAL Action Head ---
        self.dual_mlp = nn.Sequential(
            nn.Linear(2 * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_dual_classes)
        )

        # # --- HAS-ACTION (binary) Head ---
        # self.action_conf_head = nn.Sequential(
        #     nn.Linear(embed_dim, 32),
        #     nn.ReLU(),
        #     nn.Dropout(0.2),
        #     nn.Linear(32, 16),
        #     nn.ReLU(),
        #     nn.Dropout(0.2),
        #     nn.Linear(16, 1),
        #     # nn.Sigmoid()  # output in [0, 1]
        # )

    def forward(self, agent_embeddings, global_embeddings, agent_mask=None):
        """
        Args:
            agent_embeddings: [B, N, D]
            global_embeddings: [B, D]
            agent_mask: [B, N]
        Returns:
            solo_logits: [B, N, num_solo_classes]
            dual_logits: [B, N, N, num_dual_classes]
            has_action_conf: [B, 1] (value ∈ [0,1])
        """
        B, N, D = agent_embeddings.shape
        device = agent_embeddings.device

        # --- Identity Embeddings ---
        agent_indices = torch.arange(N, device=device).unsqueeze(0).expand(B, N)
        agent_id_embeds = self.agent_id_embed(agent_indices)  # [B, N, D]

        # Combine embeddings
        h = agent_embeddings + agent_id_embeds

        # Add global token
        global_token = global_embeddings.unsqueeze(1)  # [B, 1, D]
        h_full = torch.cat([global_token, h], dim=1)  # [B, N+1, D]

        # --- Attention Masking ---
        if agent_mask is not None:
            full_mask = torch.cat([torch.ones((B, 1), device=device), agent_mask], dim=1)
            attn_mask = ~full_mask.bool()
        else:
            attn_mask = None

        # --- Self-Attention ---
        h_attn_full, _ = self.self_attn(
            h_full, h_full, h_full,
            key_padding_mask=attn_mask
        )

        # Remove global token
        h = h_attn_full[:, 1:, :]  # [B, N, D]

        # --- Transformer Residuals ---
        h = self.norm1(h + agent_embeddings)
        h_ffn = self.ffn(h)
        h = self.norm2(h + h_ffn)  # [B, N, D]

        # --- SOLO Action Prediction ---
        solo_logits = self.solo_mlp(h)  # [B, N, num_solo_classes]

        # --- DUAL Action Prediction ---
        h_i = h.unsqueeze(2).expand(-1, -1, N, -1)
        h_j = h.unsqueeze(1).expand(-1, N, -1, -1)
        dual_input = torch.cat([h_i, h_j], dim=-1)
        dual_logits = self.dual_mlp(dual_input)  # [B, N, N, num_dual_classes]

        # # --- HAS-ACTION Confidence ---
        # # Pool across agents (mean pooling)
        # pooled_h = h.mean(dim=1)  # [B, D]
        # has_action_conf = self.action_conf_head(pooled_h)  # [B, 1]

        return solo_logits, dual_logits#, has_action_conf

In [18]:
class AgentActionModel(nn.Module):
    def __init__(self, in_channels=2, hidden_channels=32, embed_dim=128,
                 num_heads=4, hidden_dim=128, num_solo_classes=12, num_dual_classes=27, max_agents=4, dropout=0.1):
        super(AgentActionModel, self).__init__()

        self.max_agents = max_agents
        self.embed_dim = embed_dim

        # Feature extractors for agents and global context
        self.feat_extractor = CNN_BiLSTM_Model(19)      # expects (B, T, F_agent)
        self.global_feat_extractor = CNN_BiLSTM_Model(150)  # expects (B, T, F_global)

        # Transformer relational reasoning
        self.relational_model = TransformerRelationalModel(
            embed_dim=embed_dim,
            num_heads=num_heads,
            hidden_dim=hidden_dim,
            num_solo_classes=num_solo_classes,
            num_dual_classes=num_dual_classes,
            max_agents=max_agents,
            dropout=dropout
        )

    def forward(self, m1_feats, m2_feats, m3_feats, m4_feats,
                global_feats,
                m1_mask=None, m2_mask=None, m3_mask=None, m4_mask=None, global_mask=None):
        """
        Each agent's features: [B, T, F]
        Each agent's mask:     [B, T, F]  (1 = valid, 0 = missing)
        global_feats:          [B, T, F_global]
        """

        # --- Feature-level masking handled inside CNN_BiLSTM_Model ---
        m1_embed = self.feat_extractor(m1_feats, m1_mask)
        m2_embed = self.feat_extractor(m2_feats, m2_mask)
        m3_embed = self.feat_extractor(m3_feats, m3_mask)
        m4_embed = self.feat_extractor(m4_feats, m4_mask)
        global_embed = self.global_feat_extractor(global_feats, global_mask)

        # Stack all agent embeddings: [B, N=4, D]
        agent_embeddings = torch.stack([m1_embed, m2_embed, m3_embed, m4_embed], dim=1)
        # print("here")
        # print(agent_embeddings.shape, m1_embed.shape)

        # --- Agent-level mask ---
        # Mark agent as valid if it had any non-missing features
        agent_mask = []
        for m in [m1_mask, m2_mask, m3_mask, m4_mask]:
            if m is not None:
                valid = (m.sum(dim=(1, 2)) > 0).float()  # [B] (1 if any feature defined)
            else:
                valid = torch.ones(agent_embeddings.size(0), device=agent_embeddings.device)
            agent_mask.append(valid)

        agent_mask = torch.stack(agent_mask, dim=1)  # [B, N]

        # --- Relational reasoning with Transformer ---
        solo_logits, dual_logits = self.relational_model(agent_embeddings, global_embed, agent_mask)

        # (optional) solo_logits head can be added similarly if you define it inside TransformerRelationalModel
        return solo_logits, dual_logits

In [19]:
from sklearn.utils import resample
from collections import defaultdict
import random
import copy

def stratified_resample(chunks, min_samples_per_class, max_samples_per_class):
    label_to_chunks = defaultdict(list)

    # Group chunks by label
    for chunk in chunks:
        label_to_chunks[chunk['label']].append(chunk)

    print("Original class distribution:")
    for label, samples in label_to_chunks.items():
        print(f"  Class '{label}': {len(samples)} chunks")

    resampled_chunks = []
    new_label_to_chunks = defaultdict(list)

    for label, samples in label_to_chunks.items():
        n_samples = len(samples)

        # Special handling for "no_behavior"
        # effective_max = 10000 if label == "no_behavior" else max_samples_per_class
        effective_max = max_samples_per_class

        # --- CASE 1: Underrepresented class ---
        if n_samples < min_samples_per_class:
            augmented = []

            # Step 1: Generate rotated variants for each chunk
            rotation_angles = [90, 180, 270]
            for chunk in samples:
                for angle in rotation_angles:
                    new_chunk = copy.deepcopy(chunk)
                    new_chunk["angle"] = angle
                    augmented.append(new_chunk)

            # Combine originals + augmented
            all_possible = samples + augmented

            # Step 2: If we have enough unique variants, pick min_samples_per_class
            if len(all_possible) >= min_samples_per_class:
                resampled = random.sample(all_possible, min_samples_per_class)
            else:
                # Step 3: If still short, oversample from this pool
                resampled = resample(all_possible,
                                     replace=True,
                                     n_samples=min_samples_per_class,
                                     random_state=42)

        # --- CASE 2: Overrepresented class ---
        elif n_samples > effective_max:
            resampled = random.sample(samples, effective_max)

        # --- CASE 3: Just right ---
        else:
            resampled = samples

        resampled_chunks.extend(resampled)
        new_label_to_chunks[label] = resampled

    print("\nPost-resampling class distribution:")
    for label, samples in new_label_to_chunks.items():
        print(f"  Class '{label}': {len(samples)} chunks")

    random.shuffle(resampled_chunks)
    return resampled_chunks


In [20]:
def collate_fn(batch):
    # filter out Nones
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return torch.utils.data.default_collate(batch)

In [21]:
train_files, test_files = split_files_into_train_test(DATA_PATH, os.path.join(DATA_PATH, 'train.csv'))
train_chunks = build_chunks(DATA_PATH, os.path.join(DATA_PATH, "train.csv"), train_files)
train_chunks = filter_chunks_by_inactive_frames(train_chunks, 100)

train_chunks = stratified_resample(train_chunks, min_samples_per_class=1250, max_samples_per_class=2500)
# test_chunks = build_chunks_test(DATA_PATH, os.path.join(DATA_PATH, "train.csv"), test_files)
test_chunks = build_chunks(DATA_PATH, os.path.join(DATA_PATH, "train.csv"), test_files)

Building chunks with 762 files
Original class distribution:
  Class 'rear': 3410 chunks
  Class 'avoid': 500 chunks
  Class 'attack': 4628 chunks
  Class 'approach': 1588 chunks
  Class 'submit': 63 chunks
  Class 'chaseattack': 85 chunks
  Class 'chase': 429 chunks
  Class 'shepherd': 153 chunks
  Class 'sniff': 19931 chunks
  Class 'mount': 2180 chunks
  Class 'disengage': 214 chunks
  Class 'selfgroom': 834 chunks
  Class 'sniffgenital': 4223 chunks
  Class 'sniffbody': 1545 chunks
  Class 'sniffface': 846 chunks
  Class 'dominancemount': 301 chunks
  Class 'attemptmount': 88 chunks
  Class 'intromit': 632 chunks
  Class 'genitalgroom': 41 chunks
  Class 'reciprocalsniff': 541 chunks
  Class 'escape': 1061 chunks
  Class 'dominance': 294 chunks
  Class 'allogroom': 37 chunks
  Class 'ejaculate': 3 chunks
  Class 'defend': 779 chunks
  Class 'dig': 929 chunks
  Class 'rest': 232 chunks
  Class 'climb': 868 chunks
  Class 'run': 40 chunks
  Class 'dominancegroom': 47 chunks
  Class 'f

In [22]:
# train_chunks, test_chunks = train_chunks[:50], test_chunks[:50]

In [23]:
train_dataset = MABEModelDataset(DATA_PATH, solo_encoder, dual_encoder, train_chunks)
test_dataset = MABEModelDatasetEval(DATA_PATH, solo_encoder, dual_encoder, test_chunks)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
model = AgentActionModel()
model.load_state_dict(torch.load(r"C:\Coding\mabe\MABe_2022_TVAE\masked_cnn_attent_epoch7.pt"))
model = model.to("cuda")
D = 64
N = 4  # number of agents
# dataloader = DataLoader(
#     dataset,                # Your dataset
#     batch_size=1,     # Set your batch size
#     shuffle=True,      # Shuffle data at every epoch,
#     num_workers=4,
#     # collate_fn=truncate_collate_fn
# )
weights = torch.ones(len(solo_encoder.classes_))
weights = weights.to("cuda")
weights[0] = 0.01
weights2 = torch.ones(len(dual_encoder.classes_))
weights2 = weights2.to("cuda")
weights2[0] = 0.01
pos_weight = torch.tensor([0.1]).to("cuda")

solo_loss_fn = nn.CrossEntropyLoss(weight=weights)
dual_loss_fn = nn.CrossEntropyLoss(weight=weights2)
has_action_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 100

num chunks: 52813
num chunks: 4229


C:\Users\PC\AppData\Local\Temp\ipykernel_15860\2207317421.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r"C:\Coding\mabe\MABe_2022_TVA

In [24]:
# chunks = build_chunks(DATA_PATH, os.path.join(DATA_PATH, "train.csv"))
# chunks = filter_chunks_by_inactive_frames(chunks, 100)

# train_chunks, test_chunks = train_test_split(chunks, test_size=0.01, random_state=42)
# train_chunks = stratified_resample(train_chunks, min_samples_per_class=1250, max_samples_per_class=2500)
# # train_chunks, test_chunks = train_chunks[:50], test_chunks[:50]
# train_dataset = MABEModelDataset(DATA_PATH, solo_encoder, dual_encoder, train_chunks)
# test_dataset = MABEModelDataset(DATA_PATH, solo_encoder, dual_encoder, test_chunks)

# train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
# model = AgentActionModel()
# model.load_state_dict(torch.load(r"C:\Coding\mabe\MABe_2022_TVAE\masked_cnn_attent_epoch7.pt"))
# model = model.to("cuda")
# D = 64
# N = 4  # number of agents
# # dataloader = DataLoader(
# #     dataset,                # Your dataset
# #     batch_size=1,     # Set your batch size
# #     shuffle=True,      # Shuffle data at every epoch,
# #     num_workers=4,
# #     # collate_fn=truncate_collate_fn
# # )
# weights = torch.ones(len(solo_encoder.classes_))
# weights = weights.to("cuda")
# weights[0] = 0.09
# weights2 = torch.ones(len(dual_encoder.classes_))
# weights2 = weights2.to("cuda")
# weights2[0] = 0.09

# solo_loss_fn = nn.CrossEntropyLoss(weight=weights)
# dual_loss_fn = nn.CrossEntropyLoss(weight=weights2)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# epochs = 100

In [25]:
# def extract_behavior_segments(solo_labels, dual_labels, confidence, chunk_start, video_id, test_df, min_conf_threshold=0.5):
#     results = []

#     # Get the list of labeled behaviors and lab_id for this video
#     row = test_df.loc[test_df['video_id'] == video_id]
#     values = ast.literal_eval(row['behaviors_labeled'].values[0])
#     lab_id = row['lab_id'].values[0]

#     num_frames = solo_labels.shape[0]
#     num_mice = solo_labels.shape[1]
#     chunk_start = int(chunk_start)

#     # Create a boolean mask for confident frames
#     conf_mask = (confidence.squeeze() > min_conf_threshold)
#     total_frames = num_frames
#     dropped_frames = (~conf_mask).sum().item()


#     # ---- SOLO ACTIONS ----
#     for i in range(num_mice):
#         prev_label = 'no_behavior'
#         start = None
#         for f in range(num_frames):
#             # skip frames below confidence threshold
#             if not conf_mask[f]:
#                 curr_label = 'no_behavior'
#             else:
#                 curr_label = solo_labels[f, i]

#             if curr_label != prev_label:
#                 if prev_label != 'no_behavior':
#                     entry = {
#                         'lab_id': lab_id,
#                         'video_id': video_id,
#                         'agent_id': f'mouse{i+1}',
#                         'target_id': 'self',
#                         'action': prev_label,
#                         'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
#                         'start_frame': chunk_start + start,
#                         'stop_frame': chunk_start + f - 1,
#                     }
#                     entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                     if entry_str in values:
#                         results.append(entry)
#                 if curr_label != 'no_behavior':
#                     start = f
#                 prev_label = curr_label

#         # Handle last ongoing solo behavior
#         if prev_label != 'no_behavior':
#             entry = {
#                 'lab_id': lab_id,
#                 'video_id': video_id,
#                 'agent_id': f'mouse{i+1}',
#                 'target_id': 'self',
#                 'action': prev_label,
#                 'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
#                 'start_frame': chunk_start + start,
#                 'stop_frame': chunk_start + num_frames - 1,
#             }
#             entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#             if entry_str in values:
#                 results.append(entry)

#     # ---- DUAL ACTIONS ----
#     for i in range(num_mice):
#         for j in range(num_mice):
#             if i == j:
#                 continue
#             prev_label = 'no_behavior'
#             start = None
#             for f in range(num_frames):
#                 # skip frames below confidence threshold
#                 if not conf_mask[f]:
#                     curr_label = 'no_behavior'
#                 else:
#                     curr_label = dual_labels[f, i, j]

#                 if curr_label != prev_label:
#                     if prev_label != 'no_behavior':
#                         entry = {
#                             'lab_id': lab_id,
#                             'video_id': video_id,
#                             'agent_id': f'mouse{i+1}',
#                             'target_id': f'mouse{j+1}',
#                             'action': prev_label,
#                             'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
#                             'start_frame': chunk_start + start,
#                             'stop_frame': chunk_start + f - 1,
#                         }
#                         entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                         if entry_str in values:
#                             results.append(entry)
#                     if curr_label != 'no_behavior':
#                         start = f
#                     prev_label = curr_label

#             # Handle last ongoing dual behavior
#             if prev_label != 'no_behavior':
#                 entry = {
#                     'lab_id': lab_id,
#                     'video_id': video_id,
#                     'agent_id': f'mouse{i+1}',
#                     'target_id': f'mouse{j+1}',
#                     'action': prev_label,
#                     'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
#                     'start_frame': chunk_start + start,
#                     'stop_frame': chunk_start + num_frames - 1,
#                 }
#                 entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                 if entry_str in values:
#                     results.append(entry)

#     return pd.DataFrame(results), dropped_frames, total_frames


def extract_behavior_segments(solo_labels, dual_labels, chunk_start, video_id, test_df):
    results = []

    # Get the list of labeled behaviors and lab_id for this video
    row = test_df.loc[test_df['video_id'] == video_id]
    values = ast.literal_eval(row['behaviors_labeled'].values[0])
    lab_id = row['lab_id'].values[0]

    num_frames = solo_labels.shape[0]
    num_mice = solo_labels.shape[1]
    chunk_start = int(chunk_start)


    # ---- SOLO ACTIONS ----
    for i in range(num_mice):
        prev_label = 'no_behavior'
        start = None
        for f in range(num_frames):
            # skip frames below confidence threshold
            curr_label = solo_labels[f, i]

            if curr_label != prev_label:
                if prev_label != 'no_behavior':
                    entry = {
                        'lab_id': lab_id,
                        'video_id': video_id,
                        'agent_id': f'mouse{i+1}',
                        'target_id': 'self',
                        'action': prev_label,
                        'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
                        'start_frame': chunk_start + start,
                        'stop_frame': chunk_start + f - 1,
                    }
                    entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
                    if entry_str in values:
                        results.append(entry)
                if curr_label != 'no_behavior':
                    start = f
                prev_label = curr_label

        # Handle last ongoing solo behavior
        if prev_label != 'no_behavior':
            entry = {
                'lab_id': lab_id,
                'video_id': video_id,
                'agent_id': f'mouse{i+1}',
                'target_id': 'self',
                'action': prev_label,
                'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
                'start_frame': chunk_start + start,
                'stop_frame': chunk_start + num_frames - 1,
            }
            entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
            if entry_str in values:
                results.append(entry)

    # ---- DUAL ACTIONS ----
    for i in range(num_mice):
        for j in range(num_mice):
            if i == j:
                continue
            prev_label = 'no_behavior'
            start = None
            for f in range(num_frames):
                curr_label = dual_labels[f, i, j]

                if curr_label != prev_label:
                    if prev_label != 'no_behavior':
                        entry = {
                            'lab_id': lab_id,
                            'video_id': video_id,
                            'agent_id': f'mouse{i+1}',
                            'target_id': f'mouse{j+1}',
                            'action': prev_label,
                            'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
                            'start_frame': chunk_start + start,
                            'stop_frame': chunk_start + f - 1,
                        }
                        entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
                        if entry_str in values:
                            results.append(entry)
                    if curr_label != 'no_behavior':
                        start = f
                    prev_label = curr_label

            # Handle last ongoing dual behavior
            if prev_label != 'no_behavior':
                entry = {
                    'lab_id': lab_id,
                    'video_id': video_id,
                    'agent_id': f'mouse{i+1}',
                    'target_id': f'mouse{j+1}',
                    'action': prev_label,
                    'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
                    'start_frame': chunk_start + start,
                    'stop_frame': chunk_start + num_frames - 1,
                }
                entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
                if entry_str in values:
                    results.append(entry)

    return pd.DataFrame(results)

In [26]:
"""F Beta customized for the data format of the MABe challenge."""

import json

from collections import defaultdict

import pandas as pd
import polars as pl


class HostVisibleError(Exception):
    pass


def single_lab_f1(lab_solution: pl.DataFrame, lab_submission: pl.DataFrame, beta: float = 1) -> float:
    label_frames: defaultdict[str, set[int]] = defaultdict(set)
    prediction_frames: defaultdict[str, set[int]] = defaultdict(set)

    for row in lab_solution.to_dicts():
        label_frames[row['label_key']].update(range(row['start_frame'], row['stop_frame']))

    for video in lab_solution['video_id'].unique():
        active_labels: str = lab_solution.filter(pl.col('video_id') == video)['behaviors_labeled'].first()  # ty: ignore
        # print(f"active labels {active_labels}")
        active_labels: set[str] = set(json.loads(active_labels))
        predicted_mouse_pairs: defaultdict[str, set[int]] = defaultdict(set)

        for row in lab_submission.filter(pl.col('video_id') == video).to_dicts():
            # Since the labels are sparse, we can't evaluate prediction keys not in the active labels.
            if ','.join([str(row['agent_id']), str(row['target_id']), row['action']]) not in active_labels:
                continue

            new_frames = set(range(row['start_frame'], row['stop_frame']))
            # Ignore truly redundant predictions.
            new_frames = new_frames.difference(prediction_frames[row['prediction_key']])
            prediction_pair = ','.join([str(row['agent_id']), str(row['target_id'])])
            if predicted_mouse_pairs[prediction_pair].intersection(new_frames):
                # A single agent can have multiple targets per frame (ex: evading all other mice) but only one action per target per frame.
                raise HostVisibleError('Multiple predictions for the same frame from one agent/target pair')
            prediction_frames[row['prediction_key']].update(new_frames)
            predicted_mouse_pairs[prediction_pair].update(new_frames)

    tps = defaultdict(int)
    fns = defaultdict(int)
    fps = defaultdict(int)
    for key, pred_frames in prediction_frames.items():
        action = key.split('_')[-1]
        matched_label_frames = label_frames[key]
        tps[action] += len(pred_frames.intersection(matched_label_frames))
        fns[action] += len(matched_label_frames.difference(pred_frames))
        fps[action] += len(pred_frames.difference(matched_label_frames))

    distinct_actions = set()
    for key, frames in label_frames.items():
        action = key.split('_')[-1]
        distinct_actions.add(action)
        if key not in prediction_frames:
            fns[action] += len(frames)

    action_f1s = []
    for action in distinct_actions:
        if tps[action] + fns[action] + fps[action] == 0:
            action_f1s.append(0)
        else:
            action_f1s.append((1 + beta**2) * tps[action] / ((1 + beta**2) * tps[action] + beta**2 * fns[action] + fps[action]))
    return sum(action_f1s) / len(action_f1s)


def mouse_fbeta(solution: pd.DataFrame, submission: pd.DataFrame, beta: float = 1) -> float:
    """
    Doctests:
    >>> solution = pd.DataFrame([
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 10, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ... ])
    >>> submission = pd.DataFrame([
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 10},
    ... ])
    >>> mouse_fbeta(solution, submission)
    1.0

    >>> solution = pd.DataFrame([
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 10, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ... ])
    >>> submission = pd.DataFrame([
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'mount', 'start_frame': 0, 'stop_frame': 10}, # Wrong action
    ... ])
    >>> mouse_fbeta(solution, submission)
    0.0

    >>> solution = pd.DataFrame([
    ...     {'video_id': 123, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 9, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ...     {'video_id': 123, 'agent_id': 1, 'target_id': 2, 'action': 'mount', 'start_frame': 15, 'stop_frame': 24, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ... ])
    >>> submission = pd.DataFrame([
    ...     {'video_id': 123, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 9},
    ... ])
    >>> "%.12f" % mouse_fbeta(solution, submission)
    '0.500000000000'

    >>> solution = pd.DataFrame([
    ...     {'video_id': 123, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 9, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ...     {'video_id': 123, 'agent_id': 1, 'target_id': 2, 'action': 'mount', 'start_frame': 15, 'stop_frame': 24, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ...     {'video_id': 345, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 9, 'lab_id': 2, 'behaviors_labeled': '["1,2,attack"]'},
    ...     {'video_id': 345, 'agent_id': 1, 'target_id': 2, 'action': 'mount', 'start_frame': 15, 'stop_frame': 24, 'lab_id': 2, 'behaviors_labeled': '["1,2,attack"]'},
    ... ])
    >>> submission = pd.DataFrame([
    ...     {'video_id': 123, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 9},
    ... ])
    >>> "%.12f" % mouse_fbeta(solution, submission)
    '0.250000000000'

    >>> # Overlapping solution events, one prediction matching both.
    >>> solution = pd.DataFrame([
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 10, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 10, 'stop_frame': 20, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ... ])
    >>> submission = pd.DataFrame([
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 20},
    ... ])
    >>> mouse_fbeta(solution, submission)
    1.0

    >>> solution = pd.DataFrame([
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 10, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 30, 'stop_frame': 40, 'lab_id': 1, 'behaviors_labeled': '["1,2,attack"]'},
    ... ])
    >>> submission = pd.DataFrame([
    ...     {'video_id': 1, 'agent_id': 1, 'target_id': 2, 'action': 'attack', 'start_frame': 0, 'stop_frame': 40},
    ... ])
    >>> mouse_fbeta(solution, submission)
    0.6666666666666666
    """
    if len(solution) == 0 or len(submission) == 0:
        raise ValueError('Missing solution or submission data')

    expected_cols = ['video_id', 'agent_id', 'target_id', 'action', 'start_frame', 'stop_frame']

    for col in expected_cols:
        if col not in solution.columns:
            raise ValueError(f'Solution is missing column {col}')
        if col not in submission.columns:
            raise ValueError(f'Submission is missing column {col}')

    solution: pl.DataFrame = pl.DataFrame(solution)
    submission: pl.DataFrame = pl.DataFrame(submission)
    assert (solution['start_frame'] <= solution['stop_frame']).all()
    assert (submission['start_frame'] <= submission['stop_frame']).all()
    solution_videos = set(solution['video_id'].unique())
    # Need to align based on video IDs as we can't rely on the row IDs for handling public/private splits.
    submission = submission.filter(pl.col('video_id').is_in(solution_videos))

    solution = solution.with_columns(
        pl.concat_str(
            [
                pl.col('video_id').cast(pl.Utf8),
                pl.col('agent_id').cast(pl.Utf8),
                pl.col('target_id').cast(pl.Utf8),
                pl.col('action'),
            ],
            separator='_',
        ).alias('label_key'),
    )
    submission = submission.with_columns(
        pl.concat_str(
            [
                pl.col('video_id').cast(pl.Utf8),
                pl.col('agent_id').cast(pl.Utf8),
                pl.col('target_id').cast(pl.Utf8),
                pl.col('action'),
            ],
            separator='_',
        ).alias('prediction_key'),
    )

    lab_scores = []
    for lab in solution['lab_id'].unique():
        lab_solution = solution.filter(pl.col('lab_id') == lab).clone()
        lab_videos = set(lab_solution['video_id'].unique())
        lab_submission = submission.filter(pl.col('video_id').is_in(lab_videos)).clone()
        lab_scores.append(single_lab_f1(lab_solution, lab_submission, beta=beta))

    return sum(lab_scores) / len(lab_scores)


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str, beta: float = 1) -> float:
    """
    F1 score for the MABe Challenge
    """
    solution = solution.drop(row_id_column_name, axis='columns', errors='ignore')
    submission = submission.drop(row_id_column_name, axis='columns', errors='ignore')
    return mouse_fbeta(solution, submission, beta=beta)

In [27]:
def evaluate(model, dataloader, test_df):
    model.eval()
    total_solo_loss = 0
    total_dual_loss = 0
    total_conf_loss = 0
    total_batches = 0
    decoded_dfs = []
    actual_dfs = []
    all_conf = []
    all_has_action = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader)):
            a, b, c, d, e, solo_labels, dual_labels, has_action, video_id, chunk_start = batch
            a, b, c, d, e = a[0], b[0], c[0], d[0], e[0]
            video_id = int(video_id[0])
            # print(int(video_id))
            chunk_start = chunk_start[0]
            has_action = has_action[0].to("cuda")
            solo_labels, dual_labels = solo_labels[0].to("cuda"), dual_labels[0].to("cuda")
            a, b, c, d, e = map(
                lambda x: x.clone().detach().to(torch.float32).to("cuda") if torch.is_tensor(x) else torch.tensor(x, dtype=torch.float32).to("cuda"),
                (a, b, c, d, e)
            )
            masks = [(~x.isnan()).float() for x in [a, b, c, d, e]]

            solo_logits, dual_logits = model(a, b, c, d, e, *masks)
            # print(has_action.shape, conf.shape, solo_logits.shape, dual_logits.shape)

            solo_loss = solo_loss_fn(
                solo_logits.view(-1, len(solo_encoder.classes_)),
                solo_labels.view(-1)
            )

            mask = torch.ones_like(dual_labels, dtype=torch.bool)
            mask[:, range(N), range(N)] = False

            dual_loss = dual_loss_fn(
                dual_logits[mask],
                dual_labels[mask]
            )

            solo_preds = solo_logits.argmax(dim=-1)       # (B, 4)
            dual_preds = dual_logits.argmax(dim=-1)       # (B, 4, 4)

            total_solo_loss += solo_loss.item()
            total_dual_loss += dual_loss.item()
            total_batches += 1
            
            solo_decoded = solo_encoder.inverse_transform(
                solo_preds.cpu().numpy().reshape(-1)
            ).reshape(solo_preds.shape)

            dual_decoded = dual_encoder.inverse_transform(
                dual_preds.cpu().numpy().reshape(-1)
            ).reshape(dual_preds.shape)

            decoded_df = extract_behavior_segments(solo_decoded, dual_decoded, chunk_start, video_id, test_df)
            decoded_dfs.append(decoded_df)

            decoded_labels_solo = solo_encoder.inverse_transform(
                solo_labels.cpu().numpy().reshape(-1)
            ).reshape(solo_labels.shape)
            decoded_labels_dual = dual_encoder.inverse_transform(
                dual_labels.cpu().numpy().reshape(-1)
            ).reshape(dual_labels.shape)

            actual_df = extract_behavior_segments(decoded_labels_solo, decoded_labels_dual, chunk_start, video_id, test_df)
            actual_dfs.append(actual_df)

    predicted = pd.concat(decoded_dfs, axis=0, ignore_index=True)
    labels = pd.concat(actual_dfs, axis=0, ignore_index=True)
    if predicted.empty:
        print("PREDICTED DF EMTPY. USING DUMMY ROW")
        dummy = {
        'lab_id': labels.iloc[0]['lab_id'],
        'video_id': labels.iloc[0]['video_id'],
        'agent_id': f'mouse1',
        'target_id': 'self',
        'action': 'rear',
        'behaviors_labeled': f'["mouse1,self,rear"]',
        'start_frame': 5,
        'stop_frame': 6,
        }
        predicted = pd.concat([predicted, pd.DataFrame([dummy])], ignore_index=True)
    score = mouse_fbeta(labels, predicted)
    print(f"############ SCOORE IS {score} ########### total dropped frames")

    model.train()
    # all_conf = torch.cat(all_conf)
    # all_has_action = torch.cat(all_has_action)
    # calculate_conf_acc(all_has_action, all_conf)

    # # Compute stats
    # conf_action_1 = all_conf[all_has_action == 1]
    # conf_action_0 = all_conf[all_has_action == 0]
    # stats_action_1 = describe(conf_action_1)
    # stats_action_0 = describe(conf_action_0)

    # print("Confidence stats for has_action == 1:")
    # print(stats_action_1)
    # print("\nConfidence stats for has_action == 0:")
    # print(stats_action_0)
    return total_solo_loss / total_batches, total_dual_loss / total_batches


def describe(tensor):
    return {
        'count': len(tensor),
        'min': tensor.min().item() if len(tensor) > 0 else None,
        'max': tensor.max().item() if len(tensor) > 0 else None,
        'mean': tensor.mean().item() if len(tensor) > 0 else None,
        'std': tensor.std().item() if len(tensor) > 0 else None
    }

def calculate_conf_acc(y_true, y_score, threshold=0.5):
    y_true = y_true.cpu().numpy() 
    y_score = y_score.cpu().numpy() 
    y_pred = (y_score >= threshold).astype(int)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))

In [ ]:
for epoch in range(7, epochs):
    # Run evaluation and save every 5 epochs
    if (epoch + 1) % 1 == 0:
        test_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
        solo_eval_loss, dual_eval_loss = evaluate(model, test_loader, test_df)
        print(f"[Epoch {epoch+1}] Eval Solo Loss: {solo_eval_loss:.4f} | Eval Dual Loss: {dual_eval_loss:.4f}")

        model_path = f"masked_cnn_attent_epoch{epoch+1}.pt"
        torch.save(model.state_dict(), model_path)
        print(f"✅ Saved model at {model_path}")
        
    model.train()

    total_solo_train_loss = 0
    total_dual_train_loss = 0
    total_conf_loss = 0
    total_train_batches = 0

    for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        if batch is None:
            continue
        a, b, c, d, e, solo_labels, dual_labels, has_action = batch
        a, b, c, d, e = a[0], b[0], c[0], d[0], e[0]
        dual_labels = dual_labels[0].to("cuda")
        solo_labels = solo_labels[0].to("cuda")
        has_action = has_action[0].to("cuda")
        a, b, c, d, e = map(
            lambda x: x.clone().detach().to(torch.float32).to("cuda") if torch.is_tensor(x) else torch.tensor(x, dtype=torch.float32).to("cuda"),
            (a, b, c, d, e)
        )
        # print(a.shape, b.shape, c.shape, d.shape, e.shape, has_action.shape)
        # print(solo_labels.shape, dual_labels.shape)

        masks = [(~x.isnan()).float() for x in [a, b, c, d, e]]

        solo_logits, dual_logits = model(a, b, c, d, e, *masks)

        solo_loss = solo_loss_fn(
            solo_logits.view(-1, len(solo_encoder.classes_)),
            solo_labels.view(-1)
        )

        mask = torch.ones_like(dual_labels, dtype=torch.bool)
        mask[:, range(N), range(N)] = False

        dual_loss = dual_loss_fn(
            dual_logits[mask],
            dual_labels[mask]
        )

        # print(has_action_conf.shape)
        # has_action_loss = has_action_loss_fn(
        #     has_action_conf.squeeze(),  # [B]
        #     has_action.float().squeeze()  # [B]
        # )
        # print("solo_logits:", torch.isnan(solo_logits).any(), torch.isinf(solo_logits).any())
        # print("dual_logits:", torch.isnan(dual_logits).any(), torch.isinf(dual_logits).any())
        # print("solo_labels:", torch.isnan(solo_labels).any(), torch.isinf(solo_labels).any())
        # print("dual_labels:", torch.isnan(dual_labels).any(), torch.isinf(dual_labels).any())

        total_loss = solo_loss + dual_loss# + (has_action_loss * ALPHA)
        # print(solo_loss, dual_loss, total_loss)
        # print(f"solo={solo_loss.item():.3f}, dual={dual_loss.item():.3f}, has_action={(ALPHA * has_action_loss.item()):.3f}, total={total_loss.item():.3f}")

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        total_dual_train_loss += dual_loss.item()
        total_solo_train_loss += solo_loss.item()
        # total_conf_loss += has_action_loss.item() * ALPHA
        total_train_batches += 1

    avg_solo_train_loss = total_solo_train_loss / total_train_batches
    avg_dual_train_loss = total_dual_train_loss / total_train_batches
    # avg_conf_loss = total_conf_loss / total_train_batches

    print(f"[Epoch {epoch+1}] "
          f"Train Solo Loss: {avg_solo_train_loss:.4f} | Train Dual Loss: {avg_dual_train_loss:.4f}")


  0%|          | 0/4229 [00:00<?, ?it/s]

############ SCOORE IS 0.24140695533210904 ########### total dropped frames
[Epoch 8] Eval Solo Loss: 0.2017 | Eval Dual Loss: 1.4684
✅ Saved model at masked_cnn_attent_epoch8.pt


Epoch 8:   0%|          | 0/52813 [00:00<?, ?it/s]

[Epoch 8] Train Solo Loss: 0.4576 | Train Dual Loss: 0.7914


  0%|          | 0/4229 [00:00<?, ?it/s]

...

[Epoch 11] Train Solo Loss: 0.3990 | Train Dual Loss: 0.6739


  0%|          | 0/4229 [00:00<?, ?it/s]

############ SCOORE IS 0.2576514603468437 ########### total dropped frames
[Epoch 12] Eval Solo Loss: 0.1941 | Eval Dual Loss: 1.2483
✅ Saved model at masked_cnn_attent_epoch12.pt


Epoch 12:   0%|          | 0/52813 [00:00<?, ?it/s]

returning None


In [ ]:
# model_path = f"testing_entire_pipeline{epoch+1}.pt"
# torch.save(model.state_dict(), model_path)
# print(f"✅ Saved model at {model_path}")

✅ Saved model at testing_entire_pipeline3.pt


In [26]:
test_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
l1, l2, l3 = evaluate(model, test_loader, test_df, 0.4)
# predicted = pd.concat(decoded_dfs, axis=0, ignore_index=True)
# labels = pd.concat(actual_dfs, axis=0, ignore_index=True)
# score = mouse_fbeta(labels, predicted)
# print(score)

  0%|          | 0/2021 [00:00<?, ?it/s]

############ SCOORE IS 0.08212046560689433 ########### total dropped frames 1080387 ############### total frames 2016682 ############## Ideal score: 0.16832817984218534
Accuracy: 0.6844445480249242
Precision: 0.3609397079165284
Recall: 0.3974178186931043
Confidence stats for has_action == 1:
{'count': 487185, 'min': 0.023077912628650665, 'max': 0.966343343257904, 'mean': 0.43791815638542175, 'std': 0.2334422916173935}

Confidence stats for has_action == 0:
{'count': 1529497, 'min': 0.02182658575475216, 'max': 0.9562499523162842, 'mean': 0.314902663230896, 'std': 0.20887872576713562}


In [ ]:
# def extract_behavior_segments(solo_labels, dual_labels, confidence, chunk_start, video_id, test_df, min_conf_threshold=0.5):
#     results = []

#     # Get the list of labeled behaviors and lab_id for this video
#     row = test_df.loc[test_df['video_id'] == video_id]
#     values = ast.literal_eval(row['behaviors_labeled'].values[0])
#     lab_id = row['lab_id'].values[0]

#     num_frames = solo_labels.shape[0]
#     num_mice = solo_labels.shape[1]
#     chunk_start = int(chunk_start)

#     # Create a boolean mask for confident frames
#     conf_mask = (confidence.squeeze() > min_conf_threshold)
#     total_frames = num_frames
#     dropped_frames = (~conf_mask).sum().item()


#     # ---- SOLO ACTIONS ----
#     for i in range(num_mice):
#         prev_label = 'no_behavior'
#         start = None
#         for f in range(num_frames):
#             # skip frames below confidence threshold
#             if not conf_mask[f]:
#                 curr_label = 'no_behavior'
#             else:
#                 curr_label = solo_labels[f, i]

#             if curr_label != prev_label:
#                 if prev_label != 'no_behavior':
#                     entry = {
#                         'lab_id': lab_id,
#                         'video_id': video_id,
#                         'agent_id': f'mouse{i+1}',
#                         'target_id': 'self',
#                         'action': prev_label,
#                         'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
#                         'start_frame': chunk_start + start,
#                         'stop_frame': chunk_start + f - 1,
#                     }
#                     entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                     if entry_str in values:
#                         results.append(entry)
#                 if curr_label != 'no_behavior':
#                     start = f
#                 prev_label = curr_label

#         # Handle last ongoing solo behavior
#         if prev_label != 'no_behavior':
#             entry = {
#                 'lab_id': lab_id,
#                 'video_id': video_id,
#                 'agent_id': f'mouse{i+1}',
#                 'target_id': 'self',
#                 'action': prev_label,
#                 'behaviors_labeled': f'["mouse{i+1},self,{prev_label}"]',
#                 'start_frame': chunk_start + start,
#                 'stop_frame': chunk_start + num_frames - 1,
#             }
#             entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#             if entry_str in values:
#                 results.append(entry)

#     # ---- DUAL ACTIONS ----
#     for i in range(num_mice):
#         for j in range(num_mice):
#             if i == j:
#                 continue
#             prev_label = 'no_behavior'
#             start = None
#             for f in range(num_frames):
#                 # skip frames below confidence threshold
#                 if not conf_mask[f]:
#                     curr_label = 'no_behavior'
#                 else:
#                     curr_label = dual_labels[f, i, j]

#                 if curr_label != prev_label:
#                     if prev_label != 'no_behavior':
#                         entry = {
#                             'lab_id': lab_id,
#                             'video_id': video_id,
#                             'agent_id': f'mouse{i+1}',
#                             'target_id': f'mouse{j+1}',
#                             'action': prev_label,
#                             'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
#                             'start_frame': chunk_start + start,
#                             'stop_frame': chunk_start + f - 1,
#                         }
#                         entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                         if entry_str in values:
#                             results.append(entry)
#                     if curr_label != 'no_behavior':
#                         start = f
#                     prev_label = curr_label

#             # Handle last ongoing dual behavior
#             if prev_label != 'no_behavior':
#                 entry = {
#                     'lab_id': lab_id,
#                     'video_id': video_id,
#                     'agent_id': f'mouse{i+1}',
#                     'target_id': f'mouse{j+1}',
#                     'action': prev_label,
#                     'behaviors_labeled': f'["mouse{i+1},mouse{j+1},{prev_label}"]',
#                     'start_frame': chunk_start + start,
#                     'stop_frame': chunk_start + num_frames - 1,
#                 }
#                 entry_str = f"{entry['agent_id']},{entry['target_id']},{entry['action']}"
#                 if entry_str in values:
#                     results.append(entry)

#     return pd.DataFrame(results), dropped_frames, total_frames

# def evaluate(model, dataloader, test_df):
#     model.eval()
#     total_solo_loss = 0
#     total_dual_loss = 0
#     total_batches = 0
#     decoded_dfs = []
#     total_dropped_frames = 0
#     total_total_frames = 0
#     actual_dfs = []
#     ideal_dfs = []
#     all_conf = []
#     all_has_action = []

#     with torch.no_grad():
#         for batch_idx, batch in enumerate(tqdm(dataloader)):
#             a, b, c, d, e, solo_labels, dual_labels, has_action, video_id, chunk_start = batch
#             a, b, c, d, e = a[0], b[0], c[0], d[0], e[0]
#             video_id = int(video_id[0])
#             # print(int(video_id))
#             chunk_start = chunk_start[0]
#             has_action = has_action[0].to("cuda")
#             solo_labels, dual_labels = solo_labels[0].to("cuda"), dual_labels[0].to("cuda")
#             a, b, c, d, e = map(
#                 lambda x: x.clone().detach().to(torch.float32).to("cuda") if torch.is_tensor(x) else torch.tensor(x, dtype=torch.float32).to("cuda"),
#                 (a, b, c, d, e)
#             )
#             masks = [(~x.isnan()).float() for x in [a, b, c, d, e]]

#             solo_logits, dual_logits, conf = model(a, b, c, d, e, *masks)
#             # print(has_action.shape, conf.shape, solo_logits.shape, dual_logits.shape)
#             conf = torch.sigmoid(conf)

#             solo_loss = solo_loss_fn(
#                 solo_logits.view(-1, len(solo_encoder.classes_)),
#                 solo_labels.view(-1)
#             )

#             mask = torch.ones_like(dual_labels, dtype=torch.bool)
#             mask[:, range(N), range(N)] = False

#             dual_loss = dual_loss_fn(
#                 dual_logits[mask],
#                 dual_labels[mask]
#             )
#             solo_preds = solo_logits.argmax(dim=-1)       # (B, 4)
#             dual_preds = dual_logits.argmax(dim=-1)       # (B, 4, 4)

#             total_solo_loss += solo_loss.item()
#             total_dual_loss += dual_loss.item()
#             total_batches += 1
#             solo_decoded = solo_encoder.inverse_transform(
#                 solo_preds.cpu().numpy().reshape(-1)
#             ).reshape(solo_preds.shape)

#             dual_decoded = dual_encoder.inverse_transform(
#                 dual_preds.cpu().numpy().reshape(-1)
#             ).reshape(dual_preds.shape)

#             decoded_df, dropped_frames, total_frames = extract_behavior_segments(solo_decoded, dual_decoded, conf, chunk_start, video_id, test_df, min_conf_threshold=0.5)
#             total_dropped_frames += dropped_frames
#             total_total_frames += total_frames
#             decoded_dfs.append(decoded_df)

#             decoded_labels_solo = solo_encoder.inverse_transform(
#                 solo_labels.cpu().numpy().reshape(-1)
#             ).reshape(solo_labels.shape)
#             decoded_labels_dual = dual_encoder.inverse_transform(
#                 dual_labels.cpu().numpy().reshape(-1)
#             ).reshape(dual_labels.shape)

#             actual_df, _, _ = extract_behavior_segments(decoded_labels_solo, decoded_labels_dual, has_action, chunk_start, video_id, test_df)
#             actual_dfs.append(actual_df)
#             has_action_cpu = has_action.detach().cpu().flatten()

#             ideal_df, _, _ = extract_behavior_segments(solo_decoded, dual_decoded, has_action, chunk_start, video_id, test_df, min_conf_threshold=0.5)
#             ideal_dfs.append(ideal_df)

#             all_conf.append(conf)
#             all_has_action.append(has_action_cpu)

#     predicted = pd.concat(decoded_dfs, axis=0, ignore_index=True)
#     labels = pd.concat(actual_dfs, axis=0, ignore_index=True)
#     ideal_dfs = pd.concat(ideal_dfs, axis=0, ignore_index=True)
#     score = mouse_fbeta(labels, predicted)
#     ideal_score = mouse_fbeta(labels, ideal_dfs)
#     print(f"############ SCOORE IS {score} ########### total dropped frames {total_dropped_frames} ############### total frames {total_total_frames} ############## Ideal score: {ideal_score}")

#     model.train()
#     all_conf = torch.cat(all_conf)
#     all_has_action = torch.cat(all_has_action)
#     calculate_conf_acc(all_has_action, all_conf)

#     # Compute stats
#     conf_action_1 = all_conf[all_has_action == 1]
#     conf_action_0 = all_conf[all_has_action == 0]
#     stats_action_1 = describe(conf_action_1)
#     stats_action_0 = describe(conf_action_0)

#     print("Confidence stats for has_action == 1:")
#     print(stats_action_1)
#     print("\nConfidence stats for has_action == 0:")
#     print(stats_action_0)
#     return total_solo_loss / total_batches, total_dual_loss / total_batches


# def describe(tensor):
#     return {
#         'count': len(tensor),
#         'min': tensor.min().item() if len(tensor) > 0 else None,
#         'max': tensor.max().item() if len(tensor) > 0 else None,
#         'mean': tensor.mean().item() if len(tensor) > 0 else None,
#         'std': tensor.std().item() if len(tensor) > 0 else None
#     }

# def calculate_conf_acc(y_true, y_score, threshold=0.5):
#     y_true = y_true.cpu().numpy() 
#     y_score = y_score.cpu().numpy() 
#     y_pred = (y_score >= threshold).astype(int)
#     print("Accuracy:", accuracy_score(y_true, y_pred))
#     print("Precision:", precision_score(y_true, y_pred))
#     print("Recall:", recall_score(y_true, y_pred))

In [ ]:
# test_dataset = MABEModelDatasetEval(DATA_PATH, solo_encoder, dual_encoder, test_chunks[:3])
# test_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
# test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
# l1, l2 = evaluate(model, test_loader, test_df)

num chunks: 3


  0%|          | 0/3 [00:00<?, ?it/s]

torch.Size([1010, 1]) torch.Size([1010, 1]) torch.Size([1010, 4, 12]) torch.Size([1010, 4, 4, 27])
torch.Size([1020, 1]) torch.Size([1020, 1]) torch.Size([1020, 4, 12]) torch.Size([1020, 4, 4, 27])
torch.Size([1020, 1]) torch.Size([1020, 1]) torch.Size([1020, 4, 12]) torch.Size([1020, 4, 4, 27])
(1, 8) (27, 8)
############ SCOORE IS 0.25671641791044775 ########### total dropped frames 863 ############### total frames 3050 ############## Ideal score: 0.8045112781954887
Accuracy: 0.26557377049180325
Precision: 0.06081390032007316
Recall: 0.4169278996865204
Confidence stats for has_action == 1:
{'count': 319, 'min': 0.4306740164756775, 'max': 0.846222996711731, 'mean': 0.538079023361206, 'std': 0.1053093895316124}

Confidence stats for has_action == 0:
{'count': 2731, 'min': 0.42675721645355225, 'max': 0.8891739249229431, 'mean': 0.5726920366287231, 'std': 0.11148480325937271}


In [ ]:
# test_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
# solo_eval_loss, dual_eval_loss = evaluate(model, test_loader, test_df)

  0%|          | 0/50 [00:00<?, ?it/s]

############ SCOORE IS 0.6561412443765384 ########### total dropped frames
